# CV 4대 태스크 실습 — ④추적 실습2: Lucas-Kanade Optical Flow

### 셀 1 — 패키지 설치 및 한글 폰트 설정

In [ ]:
!pip install -q opencv-python-headless matplotlib numpy
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import abc
import os
import urllib.request
from dataclasses import dataclass, field
from typing import List, Sequence, Tuple

import cv2
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np

nanum_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if os.path.exists(nanum_path):
    fm.fontManager.addfont(nanum_path)
    plt.rcParams["font.family"] = fm.FontProperties(fname=nanum_path).get_name()
plt.rcParams["axes.unicode_minus"] = False

# ============================================================================
# 이 파일도 실습1(MeanShift)과 동일하게 4개 계층으로 구성됩니다.
#   1) 도메인 계층      : 추적점 값 객체 + 추적기 추상 인터페이스
#   2) 유스케이스 계층   : 여러 프레임에 걸친 추적 반복 + 특징점 재탐색 오케스트레이션
#   3) 인프라 계층      : cv2.calcOpticalFlowPyrLK 기반 실제 추적기 구현체
#   4) 구성 루트        : Colab 실행 코드(영상 입출력, 시각화)
# ============================================================================


# ===== [도메인 계층 / Domain Layer] =====

@dataclass
class TrackedPoint:
    """한 프레임에서 추적된 특징점 하나를 나타내는 값 객체.

    Optical Flow는 "여러 개의 점"을 동시에 추적하므로, update() 호출마다
    TrackedPoint를 N개 담은 리스트를 반환한다. MeanShift(실습1)가 항상
    TrackedRegion 1개를 반환하는 것과 대비된다 — ObjectTracker 인터페이스의
    LSP 설계 노트 참고.
    """

    x: float
    y: float
    point_id: int  # 색상/궤적을 일관되게 그리기 위한 식별자(재탐색 전까지 유지)

    @property
    def as_xy(self) -> Tuple[int, int]:
        return (int(self.x), int(self.y))


@dataclass
class OpticalFlowState:
    """추적 전체 과정에서 누적되는 상태 — 유스케이스 계층이 소유/갱신한다."""

    # 프레임별 (frame_index, 추적된 점 리스트, 이전 점 리스트) 기록
    history: List[Tuple[int, Sequence[TrackedPoint], Sequence[TrackedPoint]]] = field(
        default_factory=list
    )

    def add(self, frame_idx: int, new_points: Sequence[TrackedPoint], old_points: Sequence[TrackedPoint]) -> None:
        self.history.append((frame_idx, new_points, old_points))


class ObjectTracker(abc.ABC):
    """다중 대상(여러 특징점)을 프레임 간 추적하는 알고리즘의 추상 인터페이스.

    - SRP: "추적"만 책임지며 시각화는 포함하지 않는다.
    - DIP: 유스케이스 계층은 이 추상 타입에만 의존한다.
    - OCP: 다른 특징점 추적 알고리즘으로 교체하려면 이 인터페이스를 구현하는
      새 인프라 클래스만 추가하면 된다.
    - LSP / ISP 설계 노트: 실습1의 MeanShift는 "영역 하나"를, 이 실습의
      Optical Flow는 "다중 점"을 추적한다는 본질적 차이가 있어, 두 구현을
      하나의 좁은 인터페이스(예: "항상 점 1개 반환")에 억지로 맞추면 한쪽이
      거짓 구현이 되어 LSP를 위반하게 된다. 그래서 각 파일은 "자신에게
      자연스러운 자료형의 리스트를 반환한다"는 더 일반적인 계약을 따르는
      동일한 이름의 ObjectTracker 인터페이스를 독립적으로 정의한다(ISP:
      불필요하게 큰 공용 인터페이스를 강제하지 않고, 각 예제가 필요한 만큼만
      작은 인터페이스를 둔다). 이 파일의 update()는 TrackedPoint를 N개
      담은 리스트를 반환하여 "다중 점 추적"이라는 본질을 정직하게 표현한다.
    """

    @abc.abstractmethod
    def init(self, first_frame: np.ndarray, init_region=None) -> None:
        """첫 프레임에서 추적할 특징점들을 탐색해 초기화한다.

        init_region은 이 구현에서는 사용하지 않는다(특징점은 영상 전체에서
        자동으로 탐색됨) — MeanShift와 시그니처를 맞추되 사용하지 않는 인자는
        None 기본값으로 두어 호출부를 단순하게 유지한다.
        """
        raise NotImplementedError

    @abc.abstractmethod
    def update(self, frame: np.ndarray) -> Sequence[TrackedPoint]:
        """다음 프레임에서 각 특징점의 새 위치를 계산해 반환한다."""
        raise NotImplementedError


# ===== [유스케이스 계층 / Use Case Layer] =====

class OpticalFlowTrackingUseCase:
    """다중 특징점 추적 유스케이스.

    - SRP: "프레임 반복 + 추적 호출 + 상태 누적"만 책임진다. cv2는 import하지
      않는다(DIP) — 특징점이 부족해지면 재탐색이 필요하다는 "정책"은 알지만,
      실제 재탐색(goodFeaturesToTrack)을 어떻게 수행하는지는 인프라에 위임한다.
    """

    def __init__(self, tracker: ObjectTracker, min_points: int = 15):
        self._tracker = tracker
        self._min_points = min_points

    def run(self, first_frame: np.ndarray, frame_reader, n_frames: int) -> OpticalFlowState:
        self._tracker.init(first_frame)

        state = OpticalFlowState()
        frames_seen: List[Tuple[int, np.ndarray, Sequence[TrackedPoint], Sequence[TrackedPoint]]] = []

        for i in range(n_frames):
            ok, frame = frame_reader()
            if not ok:
                break

            old_points = self._tracker.current_points if hasattr(self._tracker, "current_points") else []
            new_points = self._tracker.update(frame)
            state.add(i, new_points, old_points)
            frames_seen.append((i, frame, new_points, old_points))

            # 특징점이 많이 유실되면 새로 다시 탐색하도록 추적기에 위임
            # (재탐색 없이는 추적이 점점 빈약해짐 — 정책 결정은 유스케이스,
            #  실제 탐색 알고리즘은 인프라의 책임으로 분리)
            if len(new_points) < self._min_points:
                self._tracker.reinit(frame, seed=i)

        self.frames_seen = frames_seen
        return state


# ===== [인프라/어댑터 계층 / Infrastructure Layer] =====

class LucasKanadeOpticalFlowTracker(ObjectTracker):
    """OpenCV Lucas-Kanade 피라미드 Optical Flow 기반 다중 특징점 추적기.

    MeanShift(실습1)가 "색 분포 영역 하나"를 통째로 추적했다면, 이 구현은
    영상 전체에서 추적하기 좋은 작은 특징점(모서리) 여러 개를 뽑아 각각
    추적한다. ObjectTracker의 구체 구현이며, OCP에 따라 이 클래스를
    추가/교체해도 OpticalFlowTrackingUseCase는 수정할 필요가 없다.
    """

    def __init__(self):
        self._feature_params = dict(maxCorners=60, qualityLevel=0.15, minDistance=7, blockSize=7)
        self._lk_params = dict(
            winSize=(15, 15), maxLevel=2,
            criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03),
        )
        self._old_gray: np.ndarray = None
        self._p0: np.ndarray = None  # cv2 포맷(N,1,2)의 특징점 좌표
        self._next_id = 0
        self._point_ids: List[int] = []
        self.current_points: Sequence[TrackedPoint] = []

    def init(self, first_frame: np.ndarray, init_region=None) -> None:
        self._old_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
        # Colab 셀 3: 추적할 "특징점(코너)" 탐색 — Shi-Tomasi 코너 검출
        # MeanShift(실습1)가 "색 분포 영역 하나"를 통째로 추적했다면,
        # Optical Flow는 영상 전체에서 추적하기 좋은 작은 특징점(모서리) 여러 개를 뽑아 각각 추적한다.
        self._p0 = cv2.goodFeaturesToTrack(self._old_gray, mask=None, **self._feature_params)
        self._assign_new_ids(len(self._p0))
        self.current_points = self._to_tracked_points(self._p0)
        print(f"추적할 특징점 개수: {len(self._p0)}")

    def _assign_new_ids(self, count: int) -> None:
        self._point_ids = list(range(self._next_id, self._next_id + count))
        self._next_id += count

    def _to_tracked_points(self, pts: np.ndarray) -> List[TrackedPoint]:
        result = []
        for j, pt in enumerate(pts):
            a, b = pt.ravel()
            pid = self._point_ids[j] if j < len(self._point_ids) else j
            result.append(TrackedPoint(x=float(a), y=float(b), point_id=pid))
        return result

    def update(self, frame: np.ndarray) -> Sequence[TrackedPoint]:
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Colab 셀 4: Lucas-Kanade Optical Flow 반복 계산
        # 매 프레임, 이전 프레임의 각 특징점이 "다음 프레임의 어디로 이동했는지"를 미분 기반으로 추정한다.
        p1, status, err = cv2.calcOpticalFlowPyrLK(
            self._old_gray, frame_gray, self._p0, None, **self._lk_params
        )

        if p1 is not None:
            mask_ok = status.flatten() == 1
            good_new = p1[mask_ok]
            good_ids = [pid for pid, keep in zip(self._point_ids, mask_ok) if keep]
        else:
            good_new = np.empty((0, 1, 2), dtype=np.float32)
            good_ids = []

        self._old_gray = frame_gray.copy()
        self._p0 = good_new.reshape(-1, 1, 2)
        self._point_ids = good_ids

        self.current_points = self._to_tracked_points(self._p0)
        return self.current_points

    def reinit(self, frame: np.ndarray, seed: int) -> None:
        """특징점이 부족해졌을 때 다시 탐색한다(유스케이스가 정책적으로 호출)."""
        self._p0 = cv2.goodFeaturesToTrack(self._old_gray, mask=None, **self._feature_params)
        self._assign_new_ids(len(self._p0))
        self.current_points = self._to_tracked_points(self._p0)


# ===== [구성 루트 / Composition Root — Colab 실행 코드] =====


### 셀 2 — 샘플 영상 준비 (실습1과 동일한 OpenCV 공식 보행자 영상)

In [ ]:
VIDEO_PATH = "/content/vtest.avi"
if not os.path.exists(VIDEO_PATH):
    url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi"
    urllib.request.urlretrieve(url, VIDEO_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
ok, old_frame = cap.read()
if not ok:
    raise RuntimeError("영상을 열 수 없습니다.")

N_FRAMES = 45

# 구체 어댑터(인프라) 생성 → 유스케이스에 추상 타입으로 주입(DIP)
tracker: ObjectTracker = LucasKanadeOpticalFlowTracker()
use_case = OpticalFlowTrackingUseCase(tracker, min_points=15)

state = use_case.run(first_frame=old_frame, frame_reader=lambda: cap.read(), n_frames=N_FRAMES)
cap.release()
print(f"{len(use_case.frames_seen)}개 시점 캡처 완료")

# 시각화 전용 보조 자료 구성(SRP: 추적 로직과 분리된 순수 그리기 함수들)
# id -> 색상 매핑(점이 재탐색되면 id가 바뀌므로 색도 자연스럽게 바뀐다)
rng = np.random.default_rng(7)
color_table = {}


def color_for(pid: int) -> Tuple[int, int, int]:
    if pid not in color_table:
        color_table[pid] = tuple(int(c) for c in rng.integers(0, 255, 3))
    return color_table[pid]


mask_canvas = np.zeros_like(old_frame)  # 이동 궤적을 그릴 캔버스
frames_to_draw: List[Tuple[int, np.ndarray, int]] = []

for i, frame, new_points, old_points in use_case.frames_seen:
    old_by_id = {p.point_id: p for p in old_points}

    # 이동 궤적선 그리기 (각 특징점이 지나온 길)
    for p in new_points:
        if p.point_id in old_by_id:
            old_p = old_by_id[p.point_id]
            color = color_for(p.point_id)
            cv2.line(mask_canvas, p.as_xy, old_p.as_xy, color, 2)

    vis = frame.copy()
    for p in new_points:
        cv2.circle(vis, p.as_xy, 4, color_for(p.point_id), -1)
    vis = cv2.add(vis, mask_canvas)

    if i % 10 == 0 or i == len(use_case.frames_seen) - 1:
        frames_to_draw.append((i, vis.copy(), len(new_points)))

    # 재탐색이 일어난 프레임은 궤적 캔버스도 초기화(끊어진 궤적을 깨끗하게 보여줌)
    if len(new_points) < 15:
        mask_canvas = np.zeros_like(old_frame)


### 셀 5 — 결과 시각화

In [ ]:
fig, axes = plt.subplots(1, len(frames_to_draw), figsize=(4.5 * len(frames_to_draw), 4.5))
for ax, (idx, vis, n_pts) in zip(axes, frames_to_draw):
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"frame {idx} (추적점 {n_pts}개)")
    ax.axis("off")
plt.suptitle("Lucas-Kanade Optical Flow — 여러 특징점을 동시에 프레임 간 추적 (선 = 이동 궤적)", fontsize=13)
plt.tight_layout()
plt.savefig("/content/optical_flow_result.png", dpi=110, bbox_inches="tight")
plt.show()
print("저장 완료: /content/optical_flow_result.png")
